# SkyPortal corpus — exploratory data analysis (A)

This notebook measures the five flattened SkyPortal tables in
`data/interim/skyportal_corpus/`: field coverage, identity and key integrity,
datetime semantics, and the anomalies that normalisation will have to face.

It is **descriptive only**. It cleans nothing, fixes nothing, and writes no data
file. Free-text and person-name columns are never displayed in full; they are
reported as length statistics only. Decisions taken from these measurements
appear in the final cell.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/interim/skyportal_corpus").is_dir())
CORPUS = ROOT / "data/interim/skyportal_corpus"

TABLE_NAMES = ["sources", "comments", "photometry", "spectra", "followup_requests"]
EXPECTED_SHAPE = {"sources": (982, 114), "comments": (2950, 13), "photometry": (7968, 44),
                  "spectra": (1, 52), "followup_requests": (2359, 164)}

TABLES = {}
for table_name in TABLE_NAMES:
    table_path = CORPUS / f"{table_name}.parquet"
    frame = pd.read_parquet(table_path)
    if len(frame) == 0:  # zero rows is a stop condition, never a result
        raise ValueError(f"Table '{table_name}' loaded 0 rows from {table_path}")
    TABLES[table_name] = frame
    print(f"{table_name:18s} rows={len(frame):5d} columns={frame.shape[1]:4d}  <- {table_path}")

EMPTY_TOKENS = {"", "[]", "{}"}
FREE_TEXT_TABLES = {"comments", "followup_requests"}
PERSON_RE = re.compile(r"(first_name|last_name|username|contact_email|contact_phone|oauth_uid"
                       r"|author_name|reporter|discoverer|^pis$|observers|reducers|bio"
                       r"|affiliations|allocation\.pi$)", re.I)
ISO_RE = re.compile(r"^\d{4}-\d{2}-\d{2}([T ]\d{2}:\d{2}(:\d{2}(\.\d+)?)?)?$")


def has_content(series):
    """True where a cell holds real content: '', '[]', '{}' and blanks are empty."""
    def alive(value):
        if value is None:
            return False
        if isinstance(value, str):
            return value.strip() not in EMPTY_TOKENS
        if isinstance(value, float) and np.isnan(value):
            return False
        return True
    return series.map(alive)


def is_iso_text(values):
    """True when every present value looks like an ISO-8601 date or timestamp."""
    return len(values) > 0 and values.map(lambda v: bool(ISO_RE.match(v))).all()


def clip(value, limit=40):
    """Keep any displayed value short; long values are truncated on sight."""
    text = str(value)
    return f"{text[:limit]}..." if len(text) > limit else text


def summarise(table, column, series, content):
    """Describe one column under the rules fixed for this notebook."""
    if not content.any():
        return "EMPTY"
    present = series[content]
    if pd.api.types.is_bool_dtype(series):
        return " | ".join(f"{k}={v}" for k, v in present.value_counts().items())
    if pd.api.types.is_numeric_dtype(series):
        return f"min={present.min():.6g} / median={present.median():.6g} / max={present.max():.6g}"
    text = present.astype(str).str.strip()
    if is_iso_text(text):
        return f"earliest={text.min()} / latest={text.max()}"
    if table in FREE_TEXT_TABLES or PERSON_RE.search(column):
        lengths = text.map(len)
        return f"length min={lengths.min()} / median={lengths.median():.0f} / max={lengths.max()}"
    if text.nunique() <= 12:
        return " | ".join(f"{clip(k)}={v}" for k, v in text.value_counts().items())
    return " | ".join(clip(v) for v in text.drop_duplicates().head(3))


def census(table):
    """One row per column: coverage and a rule-based summary, emptiest first."""
    frame = TABLES[table]
    rows = []
    for column in frame.columns:
        series = frame[column]
        content = has_content(series)
        rows.append({"column": column, "dtype": str(series.dtype),
                     "non_null": int(series.notna().sum()),
                     "coverage_pct": round(100 * content.sum() / len(frame), 2),
                     "distinct": int(series[content].astype(str).nunique()),
                     "summary": summarise(table, column, series, content)})
    return pd.DataFrame(rows).sort_values(["coverage_pct", "column"]).reset_index(drop=True)


def census_footer(table, report):
    """One line stating how much of the table carries no usable content."""
    empty = sorted(report.loc[report["coverage_pct"] == 0, "column"])
    low = int((report["coverage_pct"] < 5).sum())
    return (f"{table}: entirely empty columns = {len(empty)} | columns under 5% content "
            f"coverage (empty included) = {low} | empty columns = {empty}")


controls = pd.DataFrame(
    [{"table": n, "expected_rows": EXPECTED_SHAPE[n][0], "observed_rows": TABLES[n].shape[0],
      "expected_cols": EXPECTED_SHAPE[n][1], "observed_cols": TABLES[n].shape[1],
      "status": "PASS" if TABLES[n].shape == EXPECTED_SHAPE[n] else "FAIL"} for n in TABLE_NAMES])
print("\nCONTROLS")
print(controls.to_string(index=False))

sources            rows=  982 columns= 114  <- /home/meneses/project_astronomical/MAFORAI/data/interim/skyportal_corpus/sources.parquet
comments           rows= 2950 columns=  13  <- /home/meneses/project_astronomical/MAFORAI/data/interim/skyportal_corpus/comments.parquet
photometry         rows= 7968 columns=  44  <- /home/meneses/project_astronomical/MAFORAI/data/interim/skyportal_corpus/photometry.parquet
spectra            rows=    1 columns=  52  <- /home/meneses/project_astronomical/MAFORAI/data/interim/skyportal_corpus/spectra.parquet
followup_requests  rows= 2359 columns= 164  <- /home/meneses/project_astronomical/MAFORAI/data/interim/skyportal_corpus/followup_requests.parquet

CONTROLS
            table  expected_rows  observed_rows  expected_cols  observed_cols status
          sources            982            982            114            114   PASS
         comments           2950           2950             13             13   PASS
       photometry           7968         

In [2]:
sources_census = census("sources")
print(census_footer("sources", sources_census))
sources_census

sources: entirely empty columns = 25 | columns under 5% content coverage (empty included) = 53 | empty columns = ['altdata', 'dec_dis', 'dec_err', 'detect_photometry_count', 'dist_nearest_source', 'e_mag_nearest_source', 'host.a', 'host.alt_name', 'host.distmpc_unc', 'host.mag_fuv', 'host.mag_nuv', 'host.mag_w1', 'host.mag_w2', 'host.mag_w3', 'host.mag_w4', 'host.magk', 'host.redshift', 'host.redshift_error', 'host.sfr_w4', 'mag_nearest_source', 'mpc_name', 'ra_dis', 'ra_err', 'score', 'tns_info.end_prop_period']


,column,dtype,non_null,coverage_pct,distinct,summary
0,altdata,object,0,0.00,0,EMPTY
1,dec_dis,object,0,0.00,0,EMPTY
2,dec_err,object,0,0.00,0,EMPTY
3,detect_photometry_count,object,0,0.00,0,EMPTY
4,dist_nearest_source,object,0,0.00,0,EMPTY
5,e_mag_nearest_source,object,0,0.00,0,EMPTY
6,host.a,object,0,0.00,0,EMPTY
7,host.alt_name,object,0,0.00,0,EMPTY
8,host.distmpc_unc,object,0,0.00,0,EMPTY
9,host.mag_fuv,object,0,0.00,0,EMPTY


In [3]:
photometry_census = census("photometry")
print(census_footer("photometry", photometry_census))
photometry_census

photometry: entirely empty columns = 2 | columns under 5% content coverage (empty included) = 22 | empty columns = ['dec_unc', 'ra_unc']


,column,dtype,non_null,coverage_pct,distinct,summary
0,dec_unc,object,0,0.00,0,EMPTY
1,ra_unc,object,0,0.00,0,EMPTY
2,altdata.Telescope,object,1,0.01,1,Colibri=1
3,altdata.energy_band,object,1,0.01,1,10 keV - 10 MeV=1
4,altdata.fluence,object,1,0.01,1,7.70e-6 erg/cm^2=1
5,altdata.redshift,object,1,0.01,1,0.764=1
6,altdata.warning,object,2,0.03,2,VT_R band is not the same as besselr=1 | VT_B band is not the same as besselb=1
7,altdata.Exposure,object,4,0.05,4,6minutes=1 | 4minutes=1 | 5minutes=1 | 14minutes=1
8,altdata.filter,object,5,0.06,3,VT_B=2 | VT_R=2 | r=1
9,altdata.comment,object,6,0.08,4,calibrated using nearby stars from USNO-...=3 | VT-R band is not the same as Bessel R=1 | VT_B band is not the same as Bessel B=1 | It is not sure Mag is in Vega but we use...=1


In [4]:
comments_census = census("comments")
print(census_footer("comments", comments_census))
comments_census

comments: entirely empty columns = 1 | columns under 5% content coverage (empty included) = 1 | empty columns = ['origin']


,column,dtype,non_null,coverage_pct,distinct,summary
0,origin,object,0,0.00,0,EMPTY
1,attachment_name,object,223,7.56,190,length min=7 / median=52 / max=54
2,author_id,int64,2950,100.00,76,min=3 / median=39 / max=189
3,bot,bool,2950,100.00,2,False=2767 | True=183
4,capture_run,object,2950,100.00,1,length min=8 / median=8 / max=8
5,created_at,object,2950,100.00,2781,earliest=2022-11-10T06:21:24.729276 / latest=2026-07-21T07:31:02.676796
6,id,int64,2950,100.00,2950,min=156 / median=2123.5 / max=3651
7,modified,object,2950,100.00,2781,earliest=2022-11-10T06:21:24.729276 / latest=2026-07-21T07:31:02.676796
8,obj_id,object,2950,100.00,351,length min=6 / median=12 / max=28
9,resourceType,object,2950,100.00,1,length min=7 / median=7 / max=7


In [5]:
followup_census = census("followup_requests")
print(census_footer("followup_requests", followup_census))
followup_census

followup_requests: entirely empty columns = 20 | columns under 5% content coverage (empty included) = 50 | empty columns = ['allocation.group.description', 'allocation.group.nickname', 'allocation.instrument.last_status_update', 'allocation.instrument.listener_classname', 'allocation.instrument.tns_id', 'allocation.instrument.treasuremap_id', 'allocation.proposal_id', 'allocation.validity_ranges', 'obj.altdata', 'obj.dec_dis', 'obj.dec_err', 'obj.detect_photometry_count', 'obj.dist_nearest_source', 'obj.e_mag_nearest_source', 'obj.mag_nearest_source', 'obj.mpc_name', 'obj.ra_dis', 'obj.ra_err', 'obj.score', 'obj.tns_info.end_prop_period']


,column,dtype,non_null,coverage_pct,distinct,summary
0,allocation.group.description,object,0,0.00,0,EMPTY
1,allocation.group.nickname,object,0,0.00,0,EMPTY
2,allocation.instrument.last_status_update,object,0,0.00,0,EMPTY
3,allocation.instrument.listener_classname,object,0,0.00,0,EMPTY
4,allocation.instrument.tns_id,object,0,0.00,0,EMPTY
5,allocation.instrument.treasuremap_id,object,0,0.00,0,EMPTY
6,allocation.proposal_id,object,0,0.00,0,EMPTY
7,allocation.validity_ranges,object,9,0.00,0,EMPTY
8,obj.altdata,object,0,0.00,0,EMPTY
9,obj.dec_dis,object,0,0.00,0,EMPTY


In [6]:
spectra_census = census("spectra")
print(census_footer("spectra", spectra_census))
spectra_census

spectra: entirely empty columns = 10 | columns under 5% content coverage (empty included) = 10 | empty columns = ['altdata.', 'annotations', 'assignment_id', 'comments', 'followup_request_id', 'origin', 'owner.bio', 'owner.contact_phone', 'owner.expiration_date', 'units']


,column,dtype,non_null,coverage_pct,distinct,summary
0,altdata.,object,1,0.0,0,EMPTY
1,annotations,object,1,0.0,0,EMPTY
2,assignment_id,object,0,0.0,0,EMPTY
3,comments,object,1,0.0,0,EMPTY
4,followup_request_id,object,0,0.0,0,EMPTY
5,origin,object,0,0.0,0,EMPTY
6,owner.bio,object,0,0.0,0,EMPTY
7,owner.contact_phone,object,0,0.0,0,EMPTY
8,owner.expiration_date,object,0,0.0,0,EMPTY
9,units,object,0,0.0,0,EMPTY


In [7]:
DIRTY_ID_RE = re.compile(r"^\s|\s$|[\t\n/]|[^\x00-\x7F]")
DETAIL_TABLES = ["comments", "photometry", "spectra", "followup_requests"]

source_ids = TABLES["sources"]["id"].astype(str)
id_set = set(source_ids)
rows = [{"check": "source identity column", "table": "sources", "subject": "id",
         "finding": f"distinct={source_ids.nunique()} / distinct after strip="
                    f"{source_ids.str.strip().nunique()} / stable="
                    f"{source_ids.nunique() == source_ids.str.strip().nunique()}"}]

for value in sorted({v for v in source_ids.unique() if DIRTY_ID_RE.search(v)}):
    rows.append({"check": "dirty identifier", "table": "sources", "subject": repr(value),
                 "finding": "carries whitespace, tab, newline, slash or non-ASCII"})

for name in DETAIL_TABLES:
    frame = TABLES[name]
    source_dir = frame["source_dir"].astype(str)
    obj_id = frame["obj_id"].astype(str)
    unmatched = sorted(set(source_dir) - id_set)
    rows.append({"check": "source_dir vs sources.id", "table": name,
                 "subject": f"distinct source_dir={source_dir.nunique()}",
                 "finding": f"values absent from sources.id={len(unmatched)} "
                            f"{[repr(v) for v in unmatched[:10]]} / detail rows failing an exact "
                            f"join={int((~source_dir.isin(id_set)).sum())}"})
    rows.append({"check": "detail keys", "table": name,
                 "subject": f"own id column 'id': unique={frame['id'].nunique() == len(frame)} "
                            f"(distinct={frame['id'].nunique()} of {len(frame)} rows)",
                 "finding": f"parent reference: 'obj_id' (and capture path 'source_dir'); "
                            f"obj_id equals source_dir on {int((obj_id == source_dir).sum())} "
                            f"of {len(frame)} rows"})

identity = pd.DataFrame(rows)
identity

,check,table,subject,finding
0,source identity column,sources,id,distinct=800 / distinct after strip=800 / stable=True
1,dirty identifier,sources,'AT2023toh\t',"carries whitespace, tab, newline, slash or non-ASCII"
2,source_dir vs sources.id,comments,distinct source_dir=351,values absent from sources.id=0 [] / detail rows failing an exact join=0
3,detail keys,comments,own id column 'id': unique=True (distinct=2950 of 2950 rows),parent reference: 'obj_id' (and capture path 'source_dir'); obj_id equals source_dir on 2950 of 2950 rows
4,source_dir vs sources.id,photometry,distinct source_dir=240,values absent from sources.id=0 [] / detail rows failing an exact join=0
5,detail keys,photometry,own id column 'id': unique=True (distinct=7968 of 7968 rows),parent reference: 'obj_id' (and capture path 'source_dir'); obj_id equals source_dir on 7968 of 7968 rows
6,source_dir vs sources.id,spectra,distinct source_dir=1,values absent from sources.id=0 [] / detail rows failing an exact join=0
7,detail keys,spectra,own id column 'id': unique=True (distinct=1 of 1 rows),parent reference: 'obj_id' (and capture path 'source_dir'); obj_id equals source_dir on 1 of 1 rows
8,source_dir vs sources.id,followup_requests,distinct source_dir=395,values absent from sources.id=0 [] / detail rows failing an exact join=0
9,detail keys,followup_requests,own id column 'id': unique=False (distinct=2339 of 2359 rows),parent reference: 'obj_id' (and capture path 'source_dir'); obj_id equals source_dir on 2339 of 2359 rows


In [8]:
ISO_TZ_RE = re.compile(r"(Z|[+-]\d{2}:?\d{2})$")
MJD_RE = re.compile(r"(^|[._])(mjd|t0)$|_mjd$", re.I)
OBSERVATION_RE = re.compile(r"(mjd|(^|\.)t0$|observed_at|discoverydate|start_date|end_date"
                            r"|payload\.date$)", re.I)

rows = []
for name in TABLE_NAMES:
    frame = TABLES[name]
    for column in frame.columns:
        series = frame[column]
        present = series[has_content(series)]
        if present.empty:
            continue
        if present.map(lambda v: isinstance(v, str)).all():
            text = present.astype(str).str.strip()
            if not is_iso_text(text):
                continue
            fmt = "ISO"
            zone = "aware" if text.map(lambda v: bool(ISO_TZ_RE.search(v))).any() else "naive"
            low, high = text.min(), text.max()
        elif MJD_RE.search(column) and pd.api.types.is_numeric_dtype(series):
            fmt, zone = "MJD", "implicit UTC, no offset stored"
            low, high = f"{present.min():.5f}", f"{present.max():.5f}"
        else:
            continue
        rows.append({"table": name, "column": column, "format": fmt, "timezone": zone,
                     "min": low, "max": high, "null_count": int(len(frame) - len(present)),
                     "meaning": "OBSERVATION_TIME" if OBSERVATION_RE.search(column) else "RECORD_TIME"})

datetimes = pd.DataFrame(rows)
print("Best candidate per table for 'when this fact became known' (causal truncation anchor):")
for name in TABLE_NAMES:
    frame = TABLES[name]
    covered = int(has_content(frame["created_at"]).sum())
    observation = datetimes[(datetimes["table"] == name) & (datetimes["meaning"] == "OBSERVATION_TIME")]
    rival = ", ".join(f"{r.column} ({len(frame) - r.null_count}/{len(frame)})"
                      for r in observation.itertuples()) or "none"
    print(f"  {name:18s} -> created_at  RECORD_TIME, populated {covered}/{len(frame)}. It is the only "
          f"column that dates every row and marks entry into SkyPortal; OBSERVATION_TIME rivals "
          f"[{rival}] date the sky event, not the knowledge, and are sparse.")
datetimes

Best candidate per table for 'when this fact became known' (causal truncation anchor):
  sources            -> created_at  RECORD_TIME, populated 982/982. It is the only column that dates every row and marks entry into SkyPortal; OBSERVATION_TIME rivals [t0 (224/982), tns_info.discoverydate (84/982)] date the sky event, not the knowledge, and are sparse.
  comments           -> created_at  RECORD_TIME, populated 2950/2950. It is the only column that dates every row and marks entry into SkyPortal; OBSERVATION_TIME rivals [none] date the sky event, not the knowledge, and are sparse.
  photometry         -> created_at  RECORD_TIME, populated 7968/7968. It is the only column that dates every row and marks entry into SkyPortal; OBSERVATION_TIME rivals [mjd (7968/7968)] date the sky event, not the knowledge, and are sparse.
  spectra            -> created_at  RECORD_TIME, populated 1/1. It is the only column that dates every row and marks entry into SkyPortal; OBSERVATION_TIME rivals [observ

,table,column,format,timezone,min,max,null_count,meaning
0,sources,modified,ISO,naive,2023-05-21T19:41:45.148510,2026-07-19T09:50:32.482435,0,RECORD_TIME
1,sources,t0,MJD,"implicit UTC, no offset stored",59893.10298,61239.46984,758,OBSERVATION_TIME
2,sources,created_at,ISO,naive,2022-11-10T02:28:36.269786,2026-07-18T11:17:50.920996,0,RECORD_TIME
3,sources,tns_info.discoverydate,ISO,naive,2024-04-02 09:17:13.344,2026-07-08 22:42:34.272,898,OBSERVATION_TIME
4,sources,host.created_at,ISO,naive,2023-03-30T15:55:30.378272,2023-03-30T15:55:44.068376,980,RECORD_TIME
5,sources,host.modified,ISO,naive,2023-03-30T15:55:30.378272,2023-03-30T15:55:44.068376,980,RECORD_TIME
6,comments,created_at,ISO,naive,2022-11-10T06:21:24.729276,2026-07-21T07:31:02.676796,0,RECORD_TIME
7,comments,modified,ISO,naive,2022-11-10T06:21:24.729276,2026-07-21T07:31:02.676796,0,RECORD_TIME
8,photometry,mjd,MJD,"implicit UTC, no offset stored",2024.00000,61582.21003,0,OBSERVATION_TIME
9,photometry,created_at,ISO,naive,2022-11-10T15:36:48.606536,2026-07-24T12:20:01.229218,0,RECORD_TIME


In [9]:
ID_RE = re.compile(r"(^|[._])id$|_id$", re.I)
SENTINELS = [-9999, -999, -99, -9, -1, 999]
CATEGORICAL = [("sources", "source_profile"), ("sources", "origin"), ("sources", "redshift_origin"),
               ("comments", "resourceType"), ("photometry", "filter"), ("photometry", "magsys"),
               ("photometry", "instrument_name"), ("photometry", "origin"),
               ("followup_requests", "allocation.instrument.name")]
STATUS_PREFIXES = ["failed to submit", "submitted for", "submitted", "deleted", "complete",
                   "rejected", "pending"]
findings = []


def add(table, column, issue, affected, detail):
    findings.append({"table": table, "column": column, "issue": issue,
                     "affected_rows": affected, "detail": detail})


def safe(table, value):
    """Never let a comments/followup value reach the output in full."""
    text = str(value)
    if table in FREE_TEXT_TABLES:
        return f"{text[:20]}..." if len(text) > 20 else text
    return clip(text, 60)


for name in TABLE_NAMES:
    frame = TABLES[name]
    for column in frame.columns:
        series, content = frame[column], has_content(frame[column])
        if not content.any():
            add(name, column, "no content in any row", len(frame), "column is entirely empty")
            continue
        present = series[content]
        types = sorted({type(v).__name__ for v in present})
        if len(types) > 1:
            add(name, column, "python type varies between rows", len(present), f"types={types}")
        if present.map(lambda v: isinstance(v, str)).all():
            text = present.astype(str)
            stripped = int((text != text.str.strip()).sum())
            if stripped:
                add(name, column, "leading or trailing whitespace", stripped, "values not shown")
            replacement = int(text.str.contains("\ufffd", regex=False).sum())
            if replacement:
                add(name, column, "contains U+FFFD", replacement, "values not shown")
        elif pd.api.types.is_numeric_dtype(series) and not pd.api.types.is_bool_dtype(series):
            if not ID_RE.search(column):  # id=999 is a real key, not a sentinel
                hits = int(present.isin(SENTINELS).sum())
                if hits:
                    add(name, column, "sentinel-like numeric value", hits,
                        f"values={sorted(set(present[present.isin(SENTINELS)]))}")
            if column.lower().split(".")[-1] in {"ra", "dec"} and (present == 0).any():
                add(name, column, "exactly zero coordinate", int((present == 0).sum()),
                    "0.0 is indistinguishable from a missing coordinate")
            if ("err" in column.lower() or "unc" in column.lower()) and (present < 0).any():
                add(name, column, "negative uncertainty", int((present < 0).sum()), "value < 0")
    duplicated = int(frame[[c for c in frame.columns if c != "id"]].astype(str).duplicated().sum())
    add(name, "<whole row>", "rows identical on every column except id", duplicated,
        "0 means no such duplicate exists")

for name, column in CATEGORICAL:
    counts = TABLES[name][column][has_content(TABLES[name][column])].astype(str).value_counts()
    for value, count in counts.items():
        add(name, column, "categorical label", int(count), f"value={safe(name, value)!r}")

status = TABLES["followup_requests"]["status"].astype(str).str.strip()
add("followup_requests", "status", "categorical column is free-form", len(status),
    f"{status.nunique()} distinct values over {len(status)} rows; error text is concatenated "
    f"into the status label")
assigned = pd.Series(False, index=status.index)
for prefix in STATUS_PREFIXES:
    match = status.str.startswith(prefix) & ~assigned
    assigned |= match
    add("followup_requests", "status", "status prefix", int(match.sum()), f"prefix={prefix!r}")
add("followup_requests", "status", "status prefix", int((~assigned).sum()), "prefix=<unclassified>")

duplicate_ids = TABLES["followup_requests"]["id"]
repeated = duplicate_ids[duplicate_ids.duplicated()].unique()
block = TABLES["followup_requests"][duplicate_ids.isin(repeated)].astype(str)
varying = [c for c in TABLES["followup_requests"].columns if block.groupby(block["id"])[c].nunique().max() > 1]
add("followup_requests", "id", "primary key is not unique", int(duplicate_ids.duplicated().sum()),
    f"{len(repeated)} ids appear twice; within a repeated id only {varying} differ")

profiles = TABLES["sources"].groupby("id")["source_profile"].nunique()
multi = set(profiles[profiles > 1].index)
block = TABLES["sources"][TABLES["sources"]["id"].isin(multi)]
compare = [c for c in TABLES["sources"].columns if c not in ("source_profile", "source_file")]
raw_diff, canonical_diff = set(), set()
for _, group in block.groupby("id"):
    for column in compare:
        values = group[column].tolist()
        if len({repr(v) for v in values}) > 1:
            raw_diff.add(column)
            canon = {json.dumps(json.loads(v), sort_keys=True) if isinstance(v, str)
                     and v[:1] in "[{" else repr(v) for v in values}
            if len(canon) > 1:
                canonical_diff.add(column)
add("sources", "id", "same source captured under several profiles", len(block),
    f"{len(multi)} ids span more than one profile, occupying {len(block)} of {len(TABLES['sources'])} rows")
add("sources", "id", "multi-profile rows differ before JSON key ordering is normalised",
    len(block), f"columns differing raw={sorted(raw_diff)}")
add("sources", "id", "multi-profile rows still differ after JSON keys are sorted", len(block),
    f"columns differing canonically={sorted(canonical_diff)}; the rest differed only by key order")

candidates = pd.DataFrame(findings).sort_values(["table", "issue", "column"]).reset_index(drop=True)
raised = set(candidates["issue"])
print("Checks that ran and raised nothing:",
      [issue for issue in ["python type varies between rows", "contains U+FFFD",
                           "negative uncertainty", "sentinel-like numeric value",
                           "exactly zero coordinate"] if issue not in raised])
candidates

Checks that ran and raised nothing: ['python type varies between rows', 'contains U+FFFD', 'negative uncertainty']


,table,column,issue,affected_rows,detail
0,comments,resourceType,categorical label,2950,value='sources'
1,comments,text,leading or trailing whitespace,551,values not shown
2,comments,origin,no content in any row,2950,column is entirely empty
3,comments,<whole row>,rows identical on every column except id,0,0 means no such duplicate exists
4,followup_requests,status,categorical column is free-form,2359,187 distinct values over 2359 rows; error text is concatenated into the status label
5,followup_requests,allocation.instrument.name,categorical label,250,value='FRAM-Auger'
6,followup_requests,allocation.instrument.name,categorical label,229,value='FRAM-CTA-N'
7,followup_requests,allocation.instrument.name,categorical label,206,value='TAROT/TRE'
8,followup_requests,allocation.instrument.name,categorical label,172,value='TRT'
9,followup_requests,allocation.instrument.name,categorical label,116,value='TAROT/TCH'


In [10]:
HISTORY_PARENT = {"redshift_history": "redshift", "summary_history": "summary"}
HISTORY_VALUE = {"redshift_history": "value", "summary_history": "summary"}
UTC_OFFSET_RE = re.compile(r"([+-]\d{2}:?\d{2}|Z)$")

unique_sources = TABLES["sources"].drop_duplicates("id")  # one row per source before parsing
history_rows = []

for field, parent in HISTORY_PARENT.items():
    carrying = unique_sources[unique_sources[field].notna()]
    parsed = [(row["id"], index, entry) for _, row in carrying.iterrows()
              for index, entry in enumerate(json.loads(row[field]))]
    frame = pd.DataFrame([{"id": sid, "entry_index": index, **entry}
                          for sid, index, entry in parsed])
    value_key, stamps = HISTORY_VALUE[field], frame["set_at_utc"]
    per_source = frame["id"].value_counts()
    offset = stamps.dropna().map(lambda v: bool(UTC_OFFSET_RE.search(v)))
    unordered = frame.groupby("id")["set_at_utc"].apply(lambda s: list(s) != sorted(s))
    current = unique_sources.set_index("id")[parent]
    current = current[has_content(current)]
    latest = frame.sort_values("set_at_utc").groupby("id")[value_key].last()
    if field == "redshift_history":  # history stores the value as text, the parent as a float
        latest, current = latest.map(lambda v: None if v is None else float(v)), current.astype(float)
    shared = latest.index.intersection(current.index)
    matches = int(sum(latest[sid] == current[sid] for sid in shared))

    measured = [("sources carrying the field", len(carrying)), ("entries parsed", len(parsed)),
                ("entries per source", f"min={per_source.min()} / median={per_source.median():.0f}"
                 f" / max={per_source.max()} / most={per_source.idxmax()!r}")]
    for key in [c for c in frame.columns if c not in ("id", "entry_index")]:
        types = pd.Series([type(e[key]).__name__ for _, _, e in parsed if key in e]).value_counts()
        measured.append((f"key '{key}'", f"present in {sum(key in e for _, _, e in parsed)} entries"
                         " | types " + " ".join(f"{t}={n}" for t, n in types.items())))
    measured += [("entries with a null value (deletion events)", int(frame[value_key].isna().sum())),
                 ("set_at_utc range", f"{stamps.min()} .. {stamps.max()}"),
                 ("set_at_utc nulls", int(stamps.isna().sum())),
                 ("set_at_utc explicit UTC offset",
                  f"present={int(offset.sum())} / absent={int((~offset).sum())}"),
                 ("sources whose entries are not in chronological order",
                  f"{int(unordered.sum())} of {len(carrying)}"),
                 ("distinct set_by_user_id", frame["set_by_user_id"].nunique())]
    if field == "summary_history":
        measured += [("is_bot", " | ".join(f"{k}={v}" for k, v in frame["is_bot"].value_counts().items())),
                     ("analysis_id non-null", int(frame["analysis_id"].notna().sum()))]
    measured += [(f"history but no current '{parent}'", len(latest.index.difference(current.index))),
                 (f"latest entry by date vs current '{parent}'",
                  f"compared={len(shared)} / matches={matches} / mismatches={len(shared) - matches}")]
    history_rows += [{"field": field, "property": prop, "value": value} for prop, value in measured]

field_history = pd.DataFrame(history_rows)
field_history

,field,property,value
0,redshift_history,sources carrying the field,60
1,redshift_history,entries parsed,83
2,redshift_history,entries per source,min=1 / median=1 / max=4 / most='2025aji'
3,redshift_history,key 'value',present in 83 entries | types str=80 NoneType=3
4,redshift_history,key 'set_at_utc',present in 83 entries | types str=83
5,redshift_history,key 'uncertainty',present in 83 entries | types NoneType=63 str=20
6,redshift_history,key 'set_by_user_id',present in 83 entries | types int=83
7,redshift_history,key 'origin',present in 33 entries | types str=33
8,redshift_history,entries with a null value (deletion events),3
9,redshift_history,set_at_utc range,2023-10-19T21:18:35.567018 .. 2026-07-15T12:16:30.504968+00:00


## Decisions taken from these measurements

These decisions were taken by reading the tables above. Each one names
the finding that motivates it and the number of rows or columns it
affects. They are applied by `scripts/skyportal/02_normalise.py`, and
their effect is measured in notebook B.

| # | Decision | Motivating finding | Scope |
|---|---|---|---|
| 1 | `sources` carries one row per source: 982 -> 800. Profile membership moves into a list column. | 182 sources are returned by more than one profile query, occupying 364 rows. | 182 sources |
| 2 | When multi-profile copies of a source differ, keep the copy carrying host enrichment. Differences confined to JSON key ordering are not differences. The host family is the dotted `host.*` columns together with `host_offset`, which is derived from `host.ra` and `host.dec` and co-varies with them. | After canonicalising JSON key order, 1 of the 182 multi-profile sources genuinely differs, in 10 host-family columns. The other 181 are identical copies. | 1 source of 182 |
| 3 | `followup_requests` keeps the row where `source_dir == obj_id`: 2,359 -> 2,339. | 20 ids appear twice. `obj_id` is identical in both copies; only the capture directory differs. The 20 mismatching rows are exactly the discarded copies. | 20 rows |
| 4 | Strip whitespace from identifiers before any join. | `'AT2023toh\t'` in sources; 17 `obj_id` values in followup_requests. | 18 rows |
| 5 | Strip leading and trailing whitespace from all text columns. | `comments.text` 551, `photometry.instrument_name` 367, `allocation.pi` 172, `sources.summary` 58, `photometry.origin` 30, and others. `'NUTTelA-TAO '` counts as an instrument distinct from `'NUTTelA-TAO'`. | ~1,200 rows |
| 6 | Drop columns with no content in any row. | 25 in sources, 20 in followup_requests, 10 in spectra, 2 in photometry, 1 in comments. Empty strings and `[]` count as no content. | 58 columns |
| 7 | Drop `comments.resourceType`. | Constant `'sources'` across all 2,950 rows; it describes the endpoint, not the data. `photometry.magsys` is also constant but is retained: it is a property of the measurements. | 1 column |
| 8 | Retain sparse columns and document their coverage. | `redshift` is present on 50 sources only, but is a real value where present. Sparse is not empty. | ~100 columns |
| 9 | Add `status_normalised` with 8 values; retain `status` unchanged. | 187 distinct status strings resolve to 7 lifecycle prefixes plus 26 rows describing a processing result rather than a request state. `submitted for` alone produces 129 distinct strings because a timestamp is embedded in the label. | 2,359 rows |
| 10 | Type all datetime columns as datetime with UTC declared explicitly. | Every ISO column is timezone-naive on disk. | all tables |
| 11 | Set `limiting_mag = -1.0` to null. | 10 rows carry a sentinel, not a magnitude. | 10 rows |
| 12 | Set out-of-range `mjd` to null and flag it in a companion column. Rows are retained. | 3 rows of `GRB240911A` carry `mjd = 2024.0`, a calendar year in a day counter. 2 rows of `ZTF23aaptsuy` carry `mjd = 61582`, which falls in June 2027, 1,452 days after their own `created_at`. The remaining 14 rows whose `mjd` exceeds `created_at` differ by under a day and are normal UTC offset. | 5 rows of 7,968 |
| 13 | Flag coordinates equal to exactly 0.0 with a boolean column. Values are not nulled. | 3 in `sources.ra`, 1 in `sources.dec`, 17 in `followup_requests`. 0.0 is a valid coordinate and also what a missing value looks like; the corpus records the ambiguity rather than resolving it. | 21 rows |
| 14 | Leave `origin` and `redshift_origin` as they are, whitespace aside. | 75 distinct values in `photometry.origin`: URLs, circular numbers, coordinates, free text. These are hand-written provenance notes. Any equivalence rule would be right in some cases and wrong in others, and a wrong value is worse than an absent one. | no change |
| 15 | `created_at` is the causal truncation anchor in all five tables. | It is populated on 100% of rows in every table. `mjd`, `t0`, `observed_at` and `tns_info.discoverydate` date the sky event rather than the moment it became known, and are sparse. | all tables |
| 16 | Field history is expanded into a sixth table, `source_field_history`, one row per change. Source: `redshift_history` and `summary_history` in `sources`. Entries with a null value are kept as deletion events. `set_at_utc` is normalised to explicit UTC. `entry_index` records the original array position. `is_bot` is retained though constant; `analysis_id` is dropped as empty. The serialised columns remain in `sources`. | `redshift_history` holds 83 dated entries across 60 sources; `summary_history` holds 1,274 across 256. Each entry carries its value, the instant it was set and the user who set it. 141 of 256 summary histories are not stored in chronological order, so array position cannot be read as recency. | 1,357 rows from 266 sources |

Checks that ran and found nothing: values whose python type varies
between rows, the Unicode replacement character U+FFFD, negative
uncertainties, and rows identical on every column except their id.